# EDA — Social Media Dataset

Análise exploratória dos 52K posts cross-platform. Objetivo: entender a estrutura dos dados, distribuições e padrões antes de definir a estratégia.

**Dataset**: Kaggle Social Media Sponsorship & Engagement (CC0) · 52,214 posts · 5 plataformas · 2 anos (maio/2023–maio/2025)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.dpi'] = 100

df = pd.read_csv('../data/social_media_dataset.csv')
df['post_date'] = pd.to_datetime(df['post_date'])
df['engagement_rate'] = (df['likes'] + df['shares'] + df['comments_count']) / df['views'].replace(0, np.nan)

print(f'Shape: {df.shape}')
print(f'\nDtypes:')
print(df.dtypes)

## 1. Visão geral do dataset

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nColunas e tipos:')
print(df.dtypes)
print(f'\nValores ausentes por coluna:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'Nenhum valor ausente')
print(f'\nis_sponsored value_counts:')
print(df['is_sponsored'].value_counts())
print(f'\nPercentual patrocinado: {df["is_sponsored"].mean()*100:.1f}%')
print(f'\nDate range: {df["post_date"].min().date()} → {df["post_date"].max().date()}')

## 2. Distribuição de posts por plataforma

In [ ]:
platform_counts = df['platform'].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(platform_counts.index, platform_counts.values, color=sns.color_palette('husl', len(platform_counts)))

for bar, count in zip(bars, platform_counts.values):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
            f'{count:,}', va='center', fontsize=10)

ax.set_xlabel('Número de posts')
ax.set_title('Distribuição de posts por plataforma', fontsize=14, fontweight='bold')
ax.set_xlim(0, platform_counts.max() * 1.12)
plt.tight_layout()
plt.show()

print(platform_counts)

## 3. Distribuição do engagement rate

Engajamento calculado como `(likes + shares + comments_count) / views`.

In [ ]:
er_clipped = df['engagement_rate'].clip(0, 1)
mean_er = er_clipped.mean()
median_er = er_clipped.median()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(er_clipped.dropna(), bins=60, color='steelblue', alpha=0.75, edgecolor='white')
ax.axvline(mean_er, color='red', linestyle='--', linewidth=2, label=f'Média: {mean_er:.3f}')
ax.axvline(median_er, color='orange', linestyle='--', linewidth=2, label=f'Mediana: {median_er:.3f}')

ax.set_xlabel('Engagement Rate')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição do Engagement Rate (clipped 0–1)', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

stats = er_clipped.describe()
print(f'Média:   {stats["mean"]:.4f}')
print(f'Mediana: {er_clipped.median():.4f}')
print(f'Std:     {stats["std"]:.4f}')
print(f'P25:     {stats["25%"]:.4f}')
print(f'P75:     {stats["75%"]:.4f}')

## 4. Orgânico vs Patrocinado

In [ ]:
sponsored_stats = df.groupby('is_sponsored').agg(
    avg_engagement_rate=('engagement_rate', 'mean'),
    avg_views=('views', 'mean'),
    total_posts=('id', 'count')
).reset_index()
sponsored_stats['label'] = sponsored_stats['is_sponsored'].map({True: 'Patrocinado', False: 'Orgânico'})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(sponsored_stats['label'], sponsored_stats['avg_engagement_rate'],
            color=['#2196F3', '#FF9800'])
axes[0].set_title('Avg Engagement Rate', fontweight='bold')
axes[0].set_ylabel('Engagement Rate')
for i, v in enumerate(sponsored_stats['avg_engagement_rate']):
    axes[0].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=11)

axes[1].bar(sponsored_stats['label'], sponsored_stats['avg_views'],
            color=['#2196F3', '#FF9800'])
axes[1].set_title('Avg Views', fontweight='bold')
axes[1].set_ylabel('Views')
for i, v in enumerate(sponsored_stats['avg_views']):
    axes[1].text(i, v + 50, f'{v:,.0f}', ha='center', fontsize=11)

plt.suptitle('Orgânico vs Patrocinado', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(sponsored_stats[['label', 'avg_engagement_rate', 'avg_views', 'total_posts']].to_string(index=False))

## 5. Tendência temporal (posts por mês)

In [ ]:
df['month'] = df['post_date'].dt.to_period('M')
monthly = df.groupby('month').agg(
    total_posts=('id', 'count'),
    avg_engagement_rate=('engagement_rate', 'mean')
).reset_index()
monthly['month_str'] = monthly['month'].astype(str)

fig, ax1 = plt.subplots(figsize=(14, 5))
color1 = 'steelblue'
color2 = 'darkorange'

ax1.bar(monthly['month_str'], monthly['total_posts'], color=color1, alpha=0.6, label='Total Posts')
ax1.set_xlabel('Mês')
ax1.set_ylabel('Total de Posts', color=color1)
ax1.tick_params(axis='x', rotation=45)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(monthly['month_str'], monthly['avg_engagement_rate'], color=color2,
         linewidth=2, marker='o', markersize=4, label='Avg Engagement Rate')
ax2.set_ylabel('Avg Engagement Rate', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Tendência temporal: posts por mês e engagement rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Distribuição por tipo de conteúdo e categoria

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ct_counts = df['content_type'].value_counts()
axes[0].barh(ct_counts.index, ct_counts.values,
             color=sns.color_palette('Set2', len(ct_counts)))
axes[0].set_title('Posts por Content Type', fontweight='bold')
axes[0].set_xlabel('Número de posts')
for i, v in enumerate(ct_counts.values):
    axes[0].text(v + 20, i, f'{v:,}', va='center', fontsize=9)

cc_counts = df['content_category'].value_counts()
axes[1].barh(cc_counts.index, cc_counts.values,
             color=sns.color_palette('Set3', len(cc_counts)))
axes[1].set_title('Posts por Content Category', fontweight='bold')
axes[1].set_xlabel('Número de posts')
for i, v in enumerate(cc_counts.values):
    axes[1].text(v + 20, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Distribuição de idiomas

In [ ]:
lang_counts = df['language'].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    lang_counts.values,
    labels=lang_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('husl', len(lang_counts))
)
for autotext in autotexts:
    autotext.set_fontsize(10)

ax.set_title('Distribuição de idiomas nos posts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(lang_counts)
print(f'\nPercentuais:')
print((lang_counts / lang_counts.sum() * 100).round(1))

## 8. Achados principais

- Dataset sintético com variância mínima no engagement_rate (~19.9% em todas as dimensões) — confirma distribuição artificial
- 5 plataformas com distribuição relativamente uniforme de posts
- 42.7% dos posts são patrocinados — proporção alta para benchmarks reais
- Sem mega-influenciadores (>1M seguidores) no dataset
- Engajamento médio de 19.9% é artificialmente alto vs benchmarks reais (1-3%)